[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiashunkang/rag-ci-tutorial/blob/main/evaluation_of_rag_with_ragas.ipynb)

# Evaluate RAG pipelines with Ragas and Langfuse

This notebook is intentionally **Colab-first**. It evaluates RAG answers with Ragas, records traces in Langfuse, and writes the evaluation scores back to those traces.

## 1. Install the reproducible Colab environment

In [1]:
%pip install -q \
    ragas==0.4.3 \
    instructor==1.17.0 \
    langfuse==4.15.6 \
    datasets==5.0.1 \
    langchain==0.3.27 \
    langchain-community==0.3.30 \
    langchain-core==0.3.80 \
    langchain-openai==0.3.35 \
    openai==2.54.0

## 2. Load secrets from Colab

Create `OPENAI_API_KEY`, `LANGFUSE_PUBLIC_KEY`, and `LANGFUSE_SECRET_KEY` under **Colab > Secrets**, then grant this notebook access. `LANGFUSE_BASE_URL` is optional and defaults to the EU cloud region. Secret values are never printed.

In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY")
os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")
os.environ["LANGFUSE_BASE_URL"] = "https://cloud.langfuse.com"
print("OpenAI and Langfuse credentials are configured.")

OpenAI and Langfuse credentials are configured.


## 3. Load the evaluation data

The FIQA sample contains a question, a generated answer, and the retrieved contexts used for that answer. Ground truth is available in the source dataset, but the three metrics below do not require it.

In [3]:
from datasets import load_dataset

fiqa_eval = load_dataset("explodinggradients/fiqa", "ragas_eval", split="baseline")
print(fiqa_eval)
fiqa_eval[0]

Dataset({
    features: ['question', 'ground_truths', 'answer', 'contexts'],
    num_rows: 30
})


{'question': 'How to deposit a cheque issued to an associate in my business into my business account?',
 'ground_truths': ["Have the check reissued to the proper payee.Just have the associate sign the back and then deposit it.  It's called a third party cheque and is perfectly legal.  I wouldn't be surprised if it has a longer hold period and, as always, you don't get the money if the cheque doesn't clear. Now, you may have problems if it's a large amount or you're not very well known at the bank.  In that case you can have the associate go to the bank and endorse it in front of the teller with some ID.  You don't even technically have to be there.  Anybody can deposit money to your account if they have the account number. He could also just deposit it in his account and write a cheque to the business."],
 'answer': '\nThe best way to deposit a cheque issued to an associate in your business into your business account is to open a business account with the bank. You will need a state-is

## 4. Configure Ragas in one place

Both single-trace and batch evaluation use Ragas 0.4's native factories and collections metrics. The OpenAI client owns the timeout and retry policy, so every metric uses the same explicit configuration.

In [4]:
from openai import AsyncOpenAI
from ragas.embeddings.base import embedding_factory
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    AnswerRelevancy,
    ContextPrecisionWithoutReference,
    Faithfulness,
)

EVALUATOR_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

openai_client = AsyncOpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    timeout=120,
    max_retries=3,
)
evaluator_llm = llm_factory(EVALUATOR_MODEL, client=openai_client)
evaluator_embeddings = embedding_factory(
    "openai", model=EMBEDDING_MODEL, client=openai_client
)

METRICS = {
    "faithfulness": Faithfulness(llm=evaluator_llm),
    "answer_relevancy": AnswerRelevancy(
        llm=evaluator_llm, embeddings=evaluator_embeddings
    ),
    "context_precision_without_reference": ContextPrecisionWithoutReference(
        llm=evaluator_llm
    ),
}
print(list(METRICS))

['faithfulness', 'answer_relevancy', 'context_precision_without_reference']


## 5. Score one trace

This path is useful when every production request should receive a score. It is blocking and incurs evaluator-model cost, so many systems run it asynchronously after returning the user response.

In [5]:
import asyncio
from langfuse import get_client

langfuse = get_client()
if not langfuse.auth_check():
    raise RuntimeError("Langfuse authentication failed. Check the three LANGFUSE secrets and region URL.")
print("Langfuse client is authenticated.")

async def score_with_ragas(query, contexts, answer):
    faithfulness, relevancy, context_precision = await asyncio.gather(
        METRICS["faithfulness"].ascore(
            user_input=query, response=answer, retrieved_contexts=list(contexts)
        ),
        METRICS["answer_relevancy"].ascore(
            user_input=query, response=answer
        ),
        METRICS["context_precision_without_reference"].ascore(
            user_input=query, response=answer, retrieved_contexts=list(contexts)
        ),
    )
    results = {
        "faithfulness": faithfulness,
        "answer_relevancy": relevancy,
        "context_precision_without_reference": context_precision,
    }
    return {
        name: {"value": float(result.value), "reason": result.reason}
        for name, result in results.items()
    }

Langfuse client is authenticated.


In [6]:
row = fiqa_eval[0]
question = row["question"]
contexts = row["contexts"]
answer = row["answer"]

with langfuse.start_as_current_observation(as_type="span", name="rag") as trace:
    trace_id = trace.trace_id
    with trace.start_as_current_observation(
        name="retrieval",
        input={"question": question},
        output={"contexts": contexts},
    ):
        pass
    with trace.start_as_current_observation(
        name="generation",
        input={"question": question, "contexts": contexts},
        output={"answer": answer},
    ):
        pass
    ragas_scores = await score_with_ragas(question, contexts, answer)

for metric_name, score in ragas_scores.items():
    langfuse.create_score(
        name=metric_name,
        value=score["value"],
        comment=score["reason"],
        trace_id=trace_id,
    )
langfuse.flush()
ragas_scores

{'faithfulness': {'value': 0.6, 'reason': None},
 'answer_relevancy': {'value': 0.9366351182538324, 'reason': None},
 'context_precision_without_reference': {'value': 0.9999999999,
  'reason': None}}

## 6. Create demo traces for batch scoring

Batch scoring is usually cheaper: periodically sample production traces, reconstruct Ragas samples from retrieval and generation observations, evaluate them, and write scores back.

In [7]:
for interaction in fiqa_eval.select(range(10, 20)):
    question = interaction["question"]
    contexts = interaction["contexts"]
    answer = interaction["answer"]
    with langfuse.start_as_current_observation(as_type="span", name="rag") as trace:
        with trace.start_as_current_observation(
            name="retrieval",
            input={"question": question},
            output={"contexts": contexts},
        ):
            pass
        with trace.start_as_current_observation(
            name="generation",
            input={"question": question, "contexts": contexts},
            output={"answer": answer},
        ):
            pass

langfuse.flush()
print("Created 10 demo traces. Langfuse indexing can take 15-30 seconds.")

Created 10 demo traces. Langfuse indexing can take 15-30 seconds.


In [8]:
import json
import time

def parse_json_if_needed(value):
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value

def get_root_traces(name, limit=50):
    filters = [
        {"type": "boolean", "column": "isRootObservation", "operator": "=", "value": True},
        {"type": "string", "column": "traceName", "operator": "=", "value": name},
    ]
    rows = []
    cursor = None
    while len(rows) < limit:
        response = langfuse.api.observations.get_many(
            limit=min(100, limit - len(rows)),
            cursor=cursor,
            fields="core,basic,io,trace_context",
            filter=json.dumps(filters),
        )
        rows.extend(response.data or [])
        cursor = getattr(response.meta, "cursor", None)
        if not cursor or not response.data:
            break
    return rows[:limit]

def get_trace_observations(trace_id):
    return langfuse.api.observations.get_many(
        trace_id=trace_id, fields="core,basic,io", limit=1000
    ).data

def wait_for_root_traces(name, minimum, timeout=90):
    deadline = time.monotonic() + timeout
    traces = []
    while time.monotonic() < deadline:
        traces = get_root_traces(name=name, limit=max(20, minimum))
        if len(traces) >= minimum:
            return traces
        time.sleep(5)
    raise TimeoutError(
        f"Only found {len(traces)} '{name}' traces after {timeout}s. Check Langfuse ingestion and region settings."
    )

In [9]:
from random import Random
from ragas import Dataset

NUM_TRACES_TO_SAMPLE = 3
traces = wait_for_root_traces("rag", minimum=NUM_TRACES_TO_SAMPLE)
traces_sample = Random(42).sample(traces, NUM_TRACES_TO_SAMPLE)

batch_rows = []
for trace in traces_sample:
    question = contexts = answer = None
    for observation in get_trace_observations(trace.trace_id):
        input_payload = parse_json_if_needed(observation.input)
        output_payload = parse_json_if_needed(observation.output)
        if observation.name == "retrieval" and isinstance(input_payload, dict) and isinstance(output_payload, dict):
            question = input_payload.get("question")
            contexts = output_payload.get("contexts")
        elif observation.name == "generation" and isinstance(output_payload, dict):
            answer = output_payload.get("answer")
    if question is None or contexts is None or answer is None:
        print(f"Skipping incomplete trace {trace.trace_id}")
        continue
    batch_rows.append({
        "user_input": question,
        "retrieved_contexts": list(contexts),
        "response": answer,
        "trace_id": trace.trace_id,
    })

if not batch_rows:
    raise RuntimeError("No complete traces were available for evaluation.")

batch_dataset = Dataset(
    name="langfuse_rag_batch",
    backend="local/csv",
    root_dir="/content/ragas_runs",
    data=batch_rows,
)
print(f"Prepared {len(batch_dataset)} traces for evaluation.")

Prepared 3 traces for evaluation.


In [10]:
from ragas import experiment

@experiment()
async def run_ragas_experiment(row):
    scores = await score_with_ragas(
        row["user_input"], row["retrieved_contexts"], row["response"]
    )
    values = {name: score["value"] for name, score in scores.items()}
    reasons = {f"{name}_reason": score["reason"] for name, score in scores.items()}
    return {**row, **values, **reasons}

batch_result = await run_ragas_experiment.arun(
    batch_dataset, name="langfuse_ragas_043"
)
result_df = batch_result.to_pandas()
if len(result_df) != len(batch_rows):
    raise RuntimeError(
        f"Ragas completed {len(result_df)}/{len(batch_rows)} rows. Check warnings above."
    )
result_df

Running experiment: 100%|██████████| 3/3 [00:11<00:00,  3.81s/it]


,user_input,retrieved_contexts,response,trace_id,faithfulness,answer_relevancy,context_precision_without_reference,faithfulness_reason,answer_relevancy_reason,context_precision_without_reference_reason
0,Sole proprietorship or LLC?,[There is some benefit to creating a corporati...,\nIt depends on your individual situation and ...,ecfeccd752d4fb2fc14a4e29d9a770c1,0.5,0.000000,1.0,None,None,None
1,How to deposit a cheque issued to an associate...,[Just have the associate sign the back and the...,\nThe best way to deposit a cheque issued to a...,7abd3f35c2a406ac12aafbde608f99c2,0.6,0.936611,1.0,None,None,None
2,financial institution wants share member break...,"[""What exactly would the financial institution...",\nThe financial institution would need to see ...,547d31f61edc6c17510aa862c44486cb,1.0,0.659147,1.0,None,None,None


In [11]:
metric_names = list(METRICS)
for _, result_row in result_df.iterrows():
    for metric_name in metric_names:
        reason = result_row.get(f"{metric_name}_reason")
        if reason is not None and reason != reason:  # normalize pandas NaN
            reason = None
        langfuse.create_score(
            name=metric_name,
            value=float(result_row[metric_name]),
            comment=reason,
            trace_id=result_row["trace_id"],
        )
langfuse.flush()
result_df

,user_input,retrieved_contexts,response,trace_id,faithfulness,answer_relevancy,context_precision_without_reference,faithfulness_reason,answer_relevancy_reason,context_precision_without_reference_reason
0,Sole proprietorship or LLC?,[There is some benefit to creating a corporati...,\nIt depends on your individual situation and ...,ecfeccd752d4fb2fc14a4e29d9a770c1,0.5,0.000000,1.0,None,None,None
1,How to deposit a cheque issued to an associate...,[Just have the associate sign the back and the...,\nThe best way to deposit a cheque issued to a...,7abd3f35c2a406ac12aafbde608f99c2,0.6,0.936611,1.0,None,None,None
2,financial institution wants share member break...,"[""What exactly would the financial institution...",\nThe financial institution would need to see ...,547d31f61edc6c17510aa862c44486cb,1.0,0.659147,1.0,None,None,None
